# NFL Quarterback Rating Analysis
Composite rating system using scraped Pro Football Reference data.  
Combines 8 key stats into a weighted 0-100 score and classifies QBs into performance tiers.

In [ ]:
import sys
sys.path.insert(0, '../qb_rating_system')

from scraper import fetch_passing_stats, filter_qualified_qbs
from rating_engine import compute_ratings, get_tier_color, STAT_WEIGHTS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

YEAR = 2024

## Load and Rate QBs

In [ ]:
df = fetch_passing_stats(YEAR)
df = filter_qualified_qbs(df, min_attempts=200)
rated_df = compute_ratings(df)

print(f'{YEAR} Season: {len(rated_df)} qualifying QBs')
rated_df[['Player', 'Tm', 'Cmp%', 'Y/A', 'TD%', 'Int%', 'ANY/A', 'Win%', 'Composite_Score', 'Tier']].head(10)

## Top 15 QBs by Composite Score

In [ ]:
top15 = rated_df.head(15).sort_values('Composite_Score')
colors = [get_tier_color(t) for t in top15['Tier']]

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(top15['Player'], top15['Composite_Score'], color=colors, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Composite Score (0-100)', fontsize=12)
ax.set_title(f'{YEAR} NFL QB Rankings - Top 15', fontsize=14, fontweight='bold')
ax.set_xlim(0, 100)

# Add score labels on bars
for bar, score in zip(bars, top15['Composite_Score']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{score}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## Tier Distribution
How many QBs fall into each performance tier?

In [ ]:
tier_order = ['Elite', 'Great', 'Average', 'Below Average', 'Poor']
tier_counts = rated_df['Tier'].value_counts().reindex(tier_order, fill_value=0)
tier_colors = [get_tier_color(t) for t in tier_order]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(tier_order, tier_counts, color=tier_colors, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Number of QBs', fontsize=12)
ax.set_title(f'{YEAR} QB Tier Distribution', fontsize=14, fontweight='bold')

# Add count labels on bars
for bar, count in zip(bars, tier_counts):
    if count > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                str(count), ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## Radar Chart: Top 5 QB Stat Profiles
Spider chart showing how each elite QB's strengths compare across all 8 rating categories.

In [ ]:
def plot_radar(df, players):
    """Plot a radar chart comparing multiple QBs across normalized stats."""
    categories = [s[0] for s in STAT_WEIGHTS]
    N = len(categories)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    color_cycle = plt.cm.Set2(np.linspace(0, 1, len(players)))

    for player, color in zip(players, color_cycle):
        row = df[df['Player'] == player].iloc[0]
        values = [row[f'{cat}_norm'] for cat in categories]
        values += values[:1]
        ax.plot(angles, values, linewidth=2, label=player, color=color)
        ax.fill(angles, values, alpha=0.1, color=color)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=10)
    ax.set_ylim(0, 1)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)
    ax.set_title(f'QB Stat Profiles - {YEAR}', size=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

top5_names = rated_df.head(5)['Player'].tolist()
plot_radar(rated_df, top5_names)

## Key Stat Relationships
Scatter plots showing how key QB metrics relate to each other and to the composite score.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: TD% vs INT%
ax = axes[0, 0]
scatter = ax.scatter(rated_df['TD%'], rated_df['Int%'],
                     c=rated_df['Composite_Score'], cmap='RdYlGn',
                     s=60, edgecolors='black', linewidth=0.5)
ax.set_xlabel('TD%')
ax.set_ylabel('INT%')
ax.set_title('TD% vs INT% (colored by score)')
plt.colorbar(scatter, ax=ax, label='Score')

# Plot 2: Y/A vs Composite Score
ax = axes[0, 1]
ax.scatter(rated_df['Y/A'], rated_df['Composite_Score'],
           s=60, alpha=0.7, color='#4a90d9', edgecolors='black', linewidth=0.5)
ax.set_xlabel('Yards per Attempt')
ax.set_ylabel('Composite Score')
ax.set_title('Yards/Attempt vs Composite Score')

# Plot 3: Cmp% vs Y/A by Tier
ax = axes[1, 0]
for tier in ['Elite', 'Great', 'Average', 'Below Average', 'Poor']:
    subset = rated_df[rated_df['Tier'] == tier]
    if not subset.empty:
        ax.scatter(subset['Cmp%'], subset['Y/A'], label=tier,
                   color=get_tier_color(tier), s=60,
                   edgecolors='black', linewidth=0.5)
ax.set_xlabel('Completion %')
ax.set_ylabel('Yards per Attempt')
ax.set_title('Completion % vs Y/A by Tier')
ax.legend(fontsize=8)

# Plot 4: Composite Score distribution
ax = axes[1, 1]
ax.hist(rated_df['Composite_Score'], bins=12, edgecolor='black',
        color='#4a90d9', alpha=0.7)
ax.set_xlabel('Composite Score')
ax.set_ylabel('Count')
ax.set_title('Score Distribution')

plt.suptitle(f'{YEAR} NFL QB Analytics', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Full Rankings Table

In [ ]:
display_cols = ['Player', 'Tm', 'Cmp%', 'Y/A', 'TD%', 'Int%', 'ANY/A', 'Win%', 'Composite_Score', 'Tier']
rated_df[display_cols].style.background_gradient(subset=['Composite_Score'], cmap='RdYlGn')